# V3 · 终极版：双重审核 + 全局推理 + 人物关系

V3 是本项目的主成果。它不再相信“一次抽取 + 字面匹配”，而是把建图拆成四道工序：

1. **强推理抽取**：让模型输出事件、时间、参与者、`cause_text`、`condition_text`、`parent_text`；
2. **规则审核**：代码层剔除空事件、自环、明显不合法的字段；
3. **LLM 审核**：把**原文**和初步结果一起回喂，执行“时序铁律 / 终点禁律 / 原文验证”；
4. **全局推理 + 人物推理**：让模型以全局视角补深层因果，并推理人物之间的直接关系；代码做精确对齐、时序校验、低置信丢弃。

> 这一版仍然不是“全自动真理机”。它的目标是：**把幻觉压到可审查、可回滚的程度**。

**代价**：一次完整运行会调用 4 次 LLM（抽取、审核、全局因果、人物关系），比 V1 慢，也更贵。

In [ ]:
# ============================================================
# 0. 环境与路径：所有密钥都从 .env 读取，Notebook 里不写死
# ============================================================
import os
from pathlib import Path

from dotenv import load_dotenv

# Notebook 可能从 notebooks/ 启动，也可能从仓库根目录启动
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "").strip()
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")

# 默认不删库。第一次跑、或想从零复现时，才在 .env 里设 RESET_DB=true
RESET_DB = os.getenv("RESET_DB", "false").lower() == "true"

if not DEEPSEEK_API_KEY:
    raise RuntimeError("缺少 DEEPSEEK_API_KEY：请复制 .env.example 为 .env 并填写。")
if not NEO4J_PASSWORD:
    raise RuntimeError("缺少 NEO4J_PASSWORD：请复制 .env.example 为 .env 并填写。")

DATA_PATH = PROJECT_ROOT / "data" / "xuanwumen_text.txt"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"找不到文本文件：{DATA_PATH}")

print("仓库根目录:", PROJECT_ROOT)
print("文本文件:", DATA_PATH)
print("Neo4j:", NEO4J_URI, "| 用户:", NEO4J_USER)
print("RESET_DB:", RESET_DB)


In [ ]:
# ============================================================
# 2. 初始化 DeepSeek
# ============================================================
# 用官方 langchain-deepseek，而不是把 OpenAI 客户端硬指到 DeepSeek：
# 前者的字段、重试和流式行为与 DeepSeek API 更匹配。
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    temperature=0,  # 抽取任务尽量确定性
)
print("LLM 初始化完成:", DEEPSEEK_MODEL)


In [ ]:
# ============================================================
# 3. 读入文本
# ============================================================
text = DATA_PATH.read_text(encoding="utf-8").strip()
print(f"文本长度：{len(text)} 字符")


## 1. 第一遍：强推理抽取

这一版要求模型给出 `cause_text` / `condition_text`，但**不要求它给出事件 ID**。
ID 由代码生成，避免“长短 ID 对不上”的问题。

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

prompt_extract = PromptTemplate.from_template(
    """你是一个高级历史事件分析助手。请从文本中抽取核心事件，并做初步逻辑推理。

对每个事件输出以下字段：
- event: 主谓短语，20 字以内
- time: 原文中的时间；没有则 null
- participants: 参与者人名数组
- cause_text: 导致该事件发生的前置事件文本，必须能在原文中找到依据；没有则 null
- condition_text: 该事件发生的前提条件，必须来自原文；没有则 null
- parent_text: 该事件所属的宏观事件，统一填 "玄武门之变"

要求：
1. 按时间顺序输出；
2. 不要编造原因，找不到依据就填 null；
3. 只输出 JSON，不要输出解释或 Markdown 代码块。

格式：
{{"events": [
  {{"event": "...", "time": "...", "participants": ["..."],
    "cause_text": "...", "condition_text": "...", "parent_text": "玄武门之变"}}
]}}

文本：
{text}"""
)

chain_extract = prompt_extract | llm | JsonOutputParser()
try:
    raw = chain_extract.invoke({"text": text})
except Exception as exc:
    raise RuntimeError("第一遍抽取没有返回合法 JSON，请检查 API Key / 模型名。") from exc

raw_events = raw.get("events", []) if isinstance(raw, dict) else []
if not isinstance(raw_events, list) or not raw_events:
    raise RuntimeError(f"抽取结果为空或格式异常：{raw!r}")

print(f"第一遍抽取到 {len(raw_events)} 个事件")


## 2. 规则审核：先把明显不合法的东西挡掉

规则审核只做“代码能确定”的事，不假装能判断历史真伪：

- `event` 不能为空；
- 事件短语过长则截断（模型偶尔会写一整段）；
- `participants` 必须是数组；
- `cause_text` 不能等于事件本身；
- 保留 `parent_text`，但只作为事件属性 `e.parent`，不建额外节点，也不参与因果。

In [ ]:
def validate_events(items):
    valid = []
    for item in items:
        if not isinstance(item, dict):
            continue
        name = str(item.get("event") or "").strip()
        if not name:
            continue
        if len(name) > 40:
            name = name[:40]

        participants = item.get("participants") or []
        if not isinstance(participants, list):
            participants = []
        participants = [str(p).strip() for p in participants if str(p).strip()]

        cause_text = str(item.get("cause_text") or "").strip() or None
        if cause_text and cause_text == name:
            cause_text = None

        valid.append({
            "event": name,
            "time": item.get("time") or None,
            "participants": sorted(set(participants)),
            "cause_text": cause_text,
            "condition_text": str(item.get("condition_text") or "").strip() or None,
            "parent_text": str(item.get("parent_text") or "玄武门之变").strip(),
        })
    return valid


valid_events = validate_events(raw_events)
if not valid_events:
    raise RuntimeError("规则审核后没有有效事件。")

print(f"规则审核后剩余 {len(valid_events)} 个事件")


## 3. LLM 审核：把原文一起给它，并立下三条铁律

这是 V3 最关键的一步。只给模型 JSON、不给原文，它只能“顺着说”；
把原文一起回喂，它才有机会发现“这个原因在原文里根本不存在”。

三条铁律：

1. **时序铁律**：原因必须早于结果；
2. **终点禁律**：不能把“李世民登基 / 李渊退位”这类最终结局当成中间事件的原因；
3. **原文验证**：找不到原文依据的原因，改为 `null`。

> 注意：审核不是“让模型再编一遍”。Prompt 明确要求“不得新增/删除事件，只修正字段”。

In [ ]:
import json

prompt_audit = PromptTemplate.from_template(
    """你是一个极其严苛的历史图谱质检员。你的任务是修正初步抽取结果中的逻辑错误。

【原文】
{original_text}

【初步抽取的事件列表】
{events_json}

【审核铁律 - 必须严格遵守】
1. 时序铁律：任何事件的 cause_text，在时间上必须早于该事件本身；如果原因发生在结果之后，立刻改为 null。
2. 终点禁律：绝对不能把故事的最终大结局（如“李世民登基”“李渊退位”）作为任何中间事件的原因。
3. 原文验证：如果 cause_text 在原文中找不到依据，或者完全是编造的，立刻改为 null。
4. 条件保留：如果某事件确实需要另一个事件作为前提，保留 condition_text；否则改为 null。
5. 不得新增事件，不得删除事件，只修正字段。

请严格输出 JSON：
{{"events": [
  {{"event": "...", "time": "...", "participants": ["..."],
    "cause_text": "...", "condition_text": "...", "parent_text": "..."}}
]}}

不要输出任何解释文字，直接输出 JSON。"""
)

chain_audit = prompt_audit | llm | JsonOutputParser()
try:
    audited = chain_audit.invoke({
        "original_text": text,
        "events_json": json.dumps(valid_events, ensure_ascii=False),
    })
except Exception as exc:
    raise RuntimeError("审核阶段没有返回合法 JSON。") from exc

audited_events = audited.get("events", []) if isinstance(audited, dict) else audited
if not isinstance(audited_events, list) or not audited_events:
    raise RuntimeError(f"审核结果为空或格式异常：{audited!r}")

final_events = validate_events(audited_events)
print(f"审核后剩余 {len(final_events)} 个事件")
for ev in final_events:
    print(f"- {ev['event']}")
    print(f"    cause: {ev['cause_text']}")
    print(f"    condition: {ev['condition_text']}")


## 4. 全局常识推理：让模型以“历史学家视角”补深层因果

审核阶段解决的是“别把结局当原因”；全局推理解决的是“原文没明说、但历史上确实存在的深层因果”。

Prompt 要求模型**逐字引用**上面的事件短语作为 `cause_event` / `effect_event`。
这样代码才能做精确对齐，而不是靠 `CONTAINS` 或字符相似度瞎猜。

In [ ]:
prompt_reason = PromptTemplate.from_template(
    """你是一个顶尖的历史学家和逻辑推理专家。下面是一段历史文本中抽取出的按时间排序的事件列表：

{events_json}

请基于历史常识和原文逻辑，找出事件之间的直接因果关系。

要求：
1. cause_event 和 effect_event 必须逐字取自上面的事件短语，不要改写；
2. 原因必须早于结果；
3. 不确定就不要输出，宁缺毋滥；
4. 最多输出 20 条关系；
5. 只输出 JSON，不要输出解释。

格式：
{{"relations": [
  {{"cause_event": "原因事件原文", "effect_event": "结果事件原文",
    "logic": "一句话理由", "confidence": 0.0}}
]}}"""
)

chain_reason = prompt_reason | llm | JsonOutputParser()
try:
    reasoning = chain_reason.invoke({
        "events_json": json.dumps(
            [{"event": e["event"], "time": e["time"]} for e in final_events],
            ensure_ascii=False,
        )
    })
except Exception as exc:
    raise RuntimeError("全局推理阶段没有返回合法 JSON。") from exc

raw_relations = reasoning.get("relations", []) if isinstance(reasoning, dict) else []
causal_relations = raw_relations if isinstance(raw_relations, list) else []
print(f"模型提出 {len(causal_relations)} 条候选因果关系")


## 5. 代码裁决：精确对齐 + 时序校验 + 低置信丢弃

`resolve_event()` 的顺序是：

1. **精确匹配**（去掉空白后逐字相等）；
2. 精确匹配失败，才用 `SequenceMatcher`，阈值 **0.86**；
3. 如果 Top-2 分数相同（说明模型给了一个模棱两可的短语），直接丢弃；
4. 仍然匹配不上，丢弃。

这一步是“宁缺毋滥”的核心：**宁可少一条边，也不把两个不同事件硬连在一起。**

In [ ]:
import re
from difflib import SequenceMatcher


def _norm(text):
    return re.sub(r"\s+", "", str(text or ""))


def resolve_event(text, ev_list, threshold=0.86):
    """把模型返回的事件短语对齐到内部事件；失败返回 (None, score)。"""
    target = _norm(text)
    if not target:
        return None, 0.0

    exact = [ev for ev in ev_list if _norm(ev["event"]) == target]
    if len(exact) == 1:
        return exact[0], 1.0

    scored = sorted(
        ((SequenceMatcher(None, target, _norm(ev["event"])).ratio(), ev) for ev in ev_list),
        key=lambda pair: pair[0],
        reverse=True,
    )
    if not scored:
        return None, 0.0

    best_score, best_event = scored[0]
    if len(scored) > 1 and abs(scored[1][0] - best_score) < 1e-9:
        return None, best_score  # 两个候选一样像，放弃
    if best_score < threshold:
        return None, best_score
    return best_event, best_score


# 先给每个事件分配稳定的内部 ID，再处理因果
for order, ev in enumerate(final_events):
    ev["_id"] = f"xuanwu_v3_{order:02d}"
    ev["order"] = order

accepted, rejected = [], []
for rel in causal_relations:
    cause, c_score = resolve_event(rel.get("cause_event"), final_events)
    effect, e_score = resolve_event(rel.get("effect_event"), final_events)

    if cause is None or effect is None:
        rejected.append((rel, "事件短语无法对齐"))
        continue
    if cause["_id"] == effect["_id"]:
        rejected.append((rel, "自环"))
        continue
    if cause["order"] >= effect["order"]:
        rejected.append((rel, "违反时序铁律"))
        continue

    try:
        confidence = float(rel.get("confidence") or 0.0)
    except (TypeError, ValueError):
        confidence = 0.0

    accepted.append({
        "cause": cause,
        "effect": effect,
        "reason": str(rel.get("logic") or "").strip(),
        "confidence": confidence,
        "match_score": round(min(c_score, e_score), 4),
    })

print(f"通过裁决 {len(accepted)} 条，丢弃 {len(rejected)} 条")
for row in rejected[:8]:
    print("  丢弃:", row[1], "|", row[0])


## 6. 人物关系推理：只允许白名单关系类型

让模型自由输出“政敌 / 心腹部将 / 君主 / 同僚”等关系，但**关系类型必须落在白名单里**。
白名单之外的一律落到 `其他`，原始标签写进 `r.label`，避免把任意字符串拼进 Cypher。

同时，人物必须已经在 `Person` 节点里（来自抽取结果），否则这条关系直接丢弃。
这样不会因为模型幻觉凭空造出一个历史人物。

In [ ]:
persons = sorted({
    p for ev in final_events for p in ev.get("participants", []) if p
})
print(f"待推理人物 {len(persons)} 人：", "、".join(persons))

prompt_person = PromptTemplate.from_template(
    """你是一个顶级的唐史专家。请根据唐初玄武门之变的历史背景，推理以下人物之间真实存在的直接关系。

人物列表：
{persons}

要求：
1. 关系类型只能从下面选择：
   父子、兄弟、君臣、夫妻、师徒、政敌、盟友、幕僚、心腹部将、姻亲、同僚、叔侄、祖孙、其他
2. 不要编造人物，不要输出自环；
3. 尽可能挖掘真实存在的直接关系，但不确定的不要输出；
4. 最多 80 条；
5. 只输出 JSON。

格式：
{{"relations": [
  {{"source": "李世民", "relation": "政敌", "target": "李建成"}}
]}}"""
)

chain_person = prompt_person | llm | JsonOutputParser()
try:
    person_result = chain_person.invoke({"persons": json.dumps(persons, ensure_ascii=False)})
except Exception as exc:
    raise RuntimeError("人物关系推理阶段没有返回合法 JSON。") from exc

raw_person_relations = person_result.get("relations", []) if isinstance(person_result, dict) else []
person_relations = raw_person_relations if isinstance(raw_person_relations, list) else []
print(f"模型提出 {len(person_relations)} 条人物关系")


In [ ]:
# ============================================================
# 3. 连接 Neo4j
# ============================================================
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Neo4j 连接成功")


## 7. 终极建图：写入因果线，并删除冗余时序线

当 `cause -> effect` 已经有一条 `CAUSED_BY` 时，原本的 `FOLLOWED_BY` 就冗余了。
V3 会在同一条因果边写入成功后，删除对应方向的 `FOLLOWED_BY`，让 Browser 里的图更干净。

注意方向：

- `CAUSED_BY`：**原因 -> 结果**；
- 删除的冗余时序线也是 **原因 -> 结果**。

原始原型里这两个方向都出现过反的写法，这是本项目最值得记住的坑之一。

In [ ]:
SAFE_RELATION_TYPES = {
    "父子", "兄弟", "君臣", "夫妻", "师徒", "政敌", "盟友",
    "幕僚", "心腹部将", "姻亲", "同僚", "叔侄", "祖孙", "其他",
}


def ensure_constraints(tx):
    tx.run("""
        CREATE CONSTRAINT event_id_unique IF NOT EXISTS
        FOR (e:Event) REQUIRE e.id IS UNIQUE
    """)
    tx.run("""
        CREATE CONSTRAINT person_name_unique IF NOT EXISTS
        FOR (p:Person) REQUIRE p.name IS UNIQUE
    """)


def reset_database(tx):
    tx.run("MATCH (n) DETACH DELETE n")


def build_v3(tx, ev_list, causal_rows, p_relations):
    # ---------- 7.1 事件、人物、时序、父级属性 ----------
    for order, ev in enumerate(ev_list):
        event_id = ev["_id"]

        tx.run("""
            MERGE (e:Event {id: $id})
            SET e.name = $name,
                e.time = $time,
                e.order = $order,
                e.source = 'llm_extracted'
        """, id=event_id, name=ev["event"], time=ev["time"], order=order)

        for person in ev["participants"]:
            tx.run("""
                MERGE (p:Person {name: $name})
                WITH p
                MATCH (e:Event {id: $event_id})
                MERGE (p)-[r:PARTICIPATED_IN]->(e)
                SET r.source = 'llm_extracted'
            """, name=person, event_id=event_id)

        if order > 0:
            tx.run("""
                MATCH (prev:Event {id: $prev_id})
                MATCH (curr:Event {id: $curr_id})
                MERGE (prev)-[r:FOLLOWED_BY]->(curr)
                SET r.source = 'text_order'
            """, prev_id=ev_list[order - 1]["_id"], curr_id=event_id)

        parent = ev.get("parent_text")
        if parent:
            # 只存属性，不建 EventGroup 节点：
            # 这样 14 个事件 + 12 个人物 = 26 个节点，和参考截图一致。
            tx.run("""
                MATCH (e:Event {id: $event_id})
                SET e.parent = $parent
            """, parent=parent, event_id=event_id)

    # ---------- 7.2 因果线：原因 -> 结果 ----------
    for row in causal_rows:
        tx.run("""
            MATCH (cause:Event {id: $cause_id})
            MATCH (effect:Event {id: $effect_id})
            MERGE (cause)-[r:CAUSED_BY]->(effect)
            SET r.source = 'llm_inferred',
                r.reason = $reason,
                r.confidence = $confidence,
                r.match_score = $match_score
        """, cause_id=row["cause"]["_id"], effect_id=row["effect"]["_id"],
             reason=row["reason"], confidence=row["confidence"],
             match_score=row["match_score"])

        # 有因果就不需要同方向的时序线
        tx.run("""
            MATCH (cause:Event {id: $cause_id})-[old:FOLLOWED_BY]->(effect:Event {id: $effect_id})
            DELETE old
        """, cause_id=row["cause"]["_id"], effect_id=row["effect"]["_id"])

    # ---------- 7.3 人物关系：白名单类型 + 已存在人物 ----------
    for rel in p_relations:
        source = str(rel.get("source") or "").strip()
        target = str(rel.get("target") or "").strip()
        label = str(rel.get("relation") or "其他").strip()
        if not source or not target or source == target:
            continue

        rel_type = label if label in SAFE_RELATION_TYPES else "其他"
        # rel_type 来自白名单，不存在注入风险
        cypher = f"""
            MATCH (a:Person {{name: $source}})
            MATCH (b:Person {{name: $target}})
            MERGE (a)-[r:`{rel_type}`]->(b)
            SET r.source = 'llm_inferred', r.label = $label
        """
        tx.run(cypher, source=source, target=target, label=label)


with driver.session() as session:
    session.execute_write(ensure_constraints)
    if RESET_DB:
        session.execute_write(reset_database)
        print("已清空数据库（RESET_DB=true）")
    session.execute_write(build_v3, final_events, accepted, person_relations)

print("V3 建图完成")


## 8. 验收：数量、方向、来源

这一段不是“看着差不多就行”，而是三条硬检查：

1. **因果方向审计**：`cause.order >= effect.order` 的记录必须是 0 条；
2. **来源审计**：每条 `CAUSED_BY` 都应该有 `r.source = 'llm_inferred'`；
3. **数量审计**：事件、人物、关系数量是否符合预期。

In [ ]:
with driver.session() as session:
    event_count = session.run(
        "MATCH (e:Event) RETURN count(e) AS c"
    ).single()["c"]
    person_count = session.run(
        "MATCH (p:Person) RETURN count(p) AS c"
    ).single()["c"]
    rel_rows = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS type, count(*) AS c
        ORDER BY c DESC
    """).data()

    bad_direction = session.run("""
        MATCH (cause:Event)-[r:CAUSED_BY]->(effect:Event)
        WHERE cause.order >= effect.order
        RETURN cause.name AS cause, effect.name AS effect, r.reason AS reason
    """).data()

    missing_source = session.run("""
        MATCH ()-[r:CAUSED_BY]->()
        WHERE r.source IS NULL
        RETURN count(r) AS c
    """).single()["c"]

print(f"Event 节点：{event_count}")
print(f"Person 节点：{person_count}")
print("关系分布：")
for row in rel_rows:
    print(f"  {row['type']}: {row['c']}")

print(f"\n因果方向违规：{len(bad_direction)} 条")
for row in bad_direction[:5]:
    print("  ", row)

print(f"缺少 source 的 CAUSED_BY：{missing_source} 条")
print("\n如果方向违规 > 0，先不要相信这张图；回到第 5 节检查对齐与时序过滤。")


In [ ]:
# ============================================================
# 9. Neo4j Browser 查询
# ============================================================
queries = [
    "MATCH p=()-[r]->() RETURN p LIMIT 100;",
    "MATCH p=(e1:Event)-[r]->(e2:Event) RETURN p LIMIT 100;",
    "MATCH p=(p1:Person)-[r]->(p2:Person) RETURN p LIMIT 100;",
    "MATCH (c:Event)-[r:CAUSED_BY]->(e:Event) WHERE c.order >= e.order RETURN c.name, e.name;",
]
for q in queries:
    print(q)


## 9. 这一版解决了什么、没解决什么

**解决了：**

- 因果方向有了代码级不变式（`cause.order < effect.order`）；
- 最终事件不再被轻易当成原因（终点禁律 + 时序过滤）；
- 事件对齐从“字符相似度”改成“精确匹配优先 + 高阈值兜底”；
- 冗余时序线会被删除，图更干净；
- 人物关系有了类型白名单，避免 Cypher 注入和类型爆炸。

**没解决：**

- 审核仍由 LLM 执行，**它仍可能漏掉幻觉**；
- 全局推理补出的因果，**没有人工标注做 ground truth**；
- 人物关系来自模型常识，可能与史实有出入；
- 没有实体链接：同一人物在不同文本中可能以别名出现；
- 没有事件共指：同一事件的不同表述不会被合并。

> 所以 V3 的因果边全部标记 `source='llm_inferred'`。
> 你可以用一条 Cypher 删除所有推理边，只保留确定性关系：
>
> ```cypher
> MATCH ()-[r:CAUSED_BY]->() WHERE r.source = 'llm_inferred' DELETE r;
> ```

In [ ]:
driver.close()
print("Neo4j 连接已关闭")
